# Train ArcFace + ConvNeXt-Tiny on Google Colab
This notebook downloads / mounts your dataset, installs dependencies (`timm`), defines a `ConvNeXt-Tiny` backbone + `ArcFace` head, and trains on Colab GPU.
Instructions: Mount Google Drive or use Kaggle API (both provided). Save this notebook as `train_arcface_colab.ipynb` and run in Colab.

In [ ]:
# 1) Install dependencies (run once)
!pip install -q timm==0.9.2 safetensors kaggle einops
# (Colab already has a CUDA-enabled torch; if you need a specific torch, uncomment and install carefully)
# !pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu118

In [ ]:
# 2) Mount Google Drive (recommended)
from google.colab import drive
drive.mount('/content/drive')
DRIVE_ROOT = '/content/drive/MyDrive/kaplumbaga_dataset'  # change as needed
import os
os.makedirs(DRIVE_ROOT, exist_ok=True)
print('Drive mounted, dataset root:', DRIVE_ROOT)

### Kaggle dataset option (alternative)
If your dataset is on Kaggle, upload `kaggle.json` to Colab or Drive and use the Kaggle API to download. Example below (uncomment and set dataset slug).

In [ ]:
# Example: download from Kaggle (optional)
# from google.colab import files
# files.upload()  # upload kaggle.json if not in Drive
# !mkdir -p ~/.kaggle
# !cp kaggle.json ~/.kaggle/
# !chmod 600 ~/.kaggle/kaggle.json
# Replace 'owner/dataset-name' with actual kaggle dataset slug
# !kaggle datasets download -d owner/dataset-name -p /content
# !unzip -q /content/dataset-name.zip -d /content/dataset

In [ ]:
# 3) Point to dataset directory - either from Drive or Kaggle unzip
# Expected layout: <root>/train/<class_x>/*.jpg and <root>/val/<class_x>/*.jpg (ImageFolder compatible)
DATA_ROOT = DRIVE_ROOT  # or '/content/dataset' if downloaded from Kaggle
TRAIN_DIR = os.path.join(DATA_ROOT, 'train')
VAL_DIR = os.path.join(DATA_ROOT, 'val')
print('Train dir exists:', os.path.exists(TRAIN_DIR))
print('Val dir exists:', os.path.exists(VAL_DIR))

In [ ]:
# 4) Imports: dataloaders, model, ArcFace head, utilities
import os, math, time
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import transforms, datasets
import timm
from tqdm.auto import tqdm
from PIL import Image
print('torch:', torch.__version__, 'cuda:', torch.cuda.is_available())

In [ ]:
# 5) Data transforms and loaders
BATCH_SIZE = 64  # lower if OOM
IMG_SIZE = 224
train_transform = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(0.1,0.1,0.1,0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])
])
val_transform = transforms.Compose([
    transforms.Resize(int(IMG_SIZE*1.15)),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])
])
train_dataset = datasets.ImageFolder(TRAIN_DIR, transform=train_transform)
val_dataset = datasets.ImageFolder(VAL_DIR, transform=val_transform)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=4, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)
num_classes = len(train_dataset.classes)
print('Classes:', num_classes, 'Train samples:', len(train_dataset))

In [ ]:
# 6) ArcFace (ArcMarginProduct) implementation and model wrapper
class ArcMarginProduct(nn.Module):
    def __init__(self, in_features, out_features, s=30.0, m=0.5, easy_margin=False):
        super().__init__()
        self.in_feat = in_features
        self.out_feat = out_features
        self.s = s
        self.m = m
        self.weight = nn.Parameter(torch.FloatTensor(out_features, in_features))
        nn.init.xavier_uniform_(self.weight)
        self.cos_m = math.cos(m)
        self.sin_m = math.sin(m)
        self.th = math.cos(math.pi - m)
        self.mm = math.sin(math.pi - m) * m
        self.easy_margin = easy_margin

    def forward(self, input, label):
        # input: (N, in_features)
        cosine = F.linear(F.normalize(input), F.normalize(self.weight))
        sine = torch.sqrt(1.0 - torch.clamp(cosine ** 2, 0, 1))
        phi = cosine * self.cos_m - sine * self.sin_m
        if self.easy_margin:
            phi = torch.where(cosine > 0, phi, cosine)
        else:
            phi = torch.where(cosine > self.th, phi, cosine - self.mm)
        one_hot = torch.zeros_like(cosine)
        one_hot.scatter_(1, label.view(-1, 1), 1.0)
        output = (one_hot * phi) + ((1.0 - one_hot) * cosine)
        output = output * self.s
        return output

class ArcFaceModel(nn.Module):
    def __init__(self, num_classes, embedding_dim=512, backbone_name='convnext_tiny', pretrained=True):
        super().__init__()
        backbone = timm.create_model(backbone_name, pretrained=pretrained, num_classes=0, global_pool='avg')
        in_feat = backbone.num_features
        self.backbone = backbone
        self.embedding = nn.Sequential(
            nn.Linear(in_feat, embedding_dim),
            nn.BatchNorm1d(embedding_dim),
            nn.ReLU(inplace=True),
        )
        self.metric_fc = ArcMarginProduct(embedding_dim, num_classes)

    def forward(self, x, labels=None):
        feats = self.backbone(x)
        emb = self.embedding(feats)
        if labels is None:
            return emb
        logits = self.metric_fc(emb, labels)
        return logits, emb

In [ ]:
# 7) Training loop (run on GPU)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)
model = ArcFaceModel(num_classes=num_classes, embedding_dim=512, backbone_name='convnext_tiny', pretrained=True).to(device)
optimizer = torch.optim.AdamW([
    {'params': model.backbone.parameters(), 'lr': 1e-5},
    {'params': model.embedding.parameters(), 'lr': 1e-4},
    {'params': model.metric_fc.parameters(), 'lr': 1e-4},
], weight_decay=5e-4)
criterion = nn.CrossEntropyLoss()
EPOCHS = 20
best_acc = 0.0
for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    loop = tqdm(train_loader, leave=False)
    for imgs, labels in loop:
        imgs = imgs.to(device)
        labels = labels.to(device)
        optimizer.zero_grad()
        logits, emb = model(imgs, labels)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * imgs.size(0)
        _, preds = logits.max(1)
        total += labels.size(0)
        correct += preds.eq(labels).sum().item()
        loop.set_postfix(loss=loss.item(), acc=100.*correct/total)
    epoch_loss = running_loss / len(train_loader.dataset)
    epoch_acc = 100. * correct / total
    print(f'Epoch {epoch+1}/{EPOCHS} Loss: {epoch_loss:.4f} Acc: {epoch_acc:.2f}%')
    # Save checkpoint to Drive
    ckpt_path = os.path.join(DRIVE_ROOT, f'arcface_convnext_epoch{epoch+1}.pth')
    torch.save({'epoch': epoch, 'state_dict': model.state_dict(), 'optimizer': optimizer.state_dict()}, ckpt_path)
    print('Saved', ckpt_path)

print('Training complete')

### Notes / Next steps
- If you hit OOM, lower `BATCH_SIZE` or reduce image `IMG_SIZE` to 160.
- To resume training: load checkpoint with `torch.load()` and `model.load_state_dict(...)`.
- You can enable `wandb` or TensorBoard for richer monitoring.
- After training, move `arcface_convnext_epochX.pth` from Drive to your local machine.